# Entraînement LoRA Ambrelune (SD1.5, kohya sd-scripts)

**Avant de lancer :**
1. Réglages du notebook (à droite) → **Accelerator : GPU T4 x2** (ou T4 simple) + **Internet : On**.
2. Dataset **Ambrelune** déjà ajouté en Input.
3. Exécuter les cellules dans l'ordre (Run All).

Durée attendue : ~30-45 min pour 45 images, 10 répétitions, 10 époques sur T4.

In [ ]:
import glob, os

# Kaggle a monte ce dataset sous /kaggle/input/datasets/gregorycuypers/ambrelune (pas
# directement /kaggle/input/ambrelune) : on cherche automatiquement le bon dossier pour
# ne plus jamais dependre de la structure exacte.
_hits = glob.glob("/kaggle/input/**/perso_chloe_face.png", recursive=True)
DATASET_DIR = os.path.dirname(_hits[0]) if _hits else "/kaggle/input/ambrelune"
print("DATASET_DIR detecte:", DATASET_DIR)

PROJECT_NAME = "ambrelune_style"
TRIGGER_WORD = "ambrelune style"  # deja present dans chaque legende
REPEATS = 10       # nb de repetitions par epoque (image x repeats vues par epoque)
EPOCHS = 10
NETWORK_DIM = 32   # rang de la LoRA (16-32 suffit pour un style)
NETWORK_ALPHA = 16
LEARNING_RATE = 1e-4
RESOLUTION = 512

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Installation de kohya sd-scripts + dependances
import os
if not os.path.isdir("/kaggle/working/sd-scripts"):
    !git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
%cd /kaggle/working/sd-scripts
!pip install -q --no-deps -r requirements.txt
# transformers==5.17.0 (pin fixe de requirements.txt) exige tokenizers 0.23.x ; la base
# Kaggle avait une version plus ancienne que --no-deps ne met pas a jour seule.
!pip install -q "tokenizers>=0.23.1,<0.24.0"

In [ ]:
# Checkpoint de base SD1.5 (celui utilise dans le workflow ComfyUI local)
import os
!mkdir -p /kaggle/working/models
_ckpt = "/kaggle/working/models/v1-5-pruned-emaonly.safetensors"
if not (os.path.isfile(_ckpt) and os.path.getsize(_ckpt) > 10**9):
    !wget -q -O {_ckpt} https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
else:
    print("deja telecharge, on saute")
!ls -la /kaggle/working/models/

In [ ]:
# Deballage/detection du dataset dans la structure attendue par kohya : <repeats>_<label>
# Gere les deux cas : Kaggle a garde le .zip tel quel, OU l'a deja decompresse a l'import.
import zipfile, os, shutil, glob

DATA_ROOT = "/kaggle/working/dataset"
IMG_DIR = f"{DATA_ROOT}/{REPEATS}_{PROJECT_NAME}"
os.makedirs(IMG_DIR, exist_ok=True)

zips = glob.glob(f"{DATASET_DIR}/**/*.zip", recursive=True)
if zips:
    print("zip trouve :", zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working/_raw")
    src_dir = "/kaggle/working/_raw"
else:
    print("pas de zip : dataset deja decompresse par Kaggle, lecture directe de", DATASET_DIR)
    src_dir = DATASET_DIR

n = 0
for root, _, files in os.walk(src_dir):
    for f in files:
        if f.endswith((".png", ".jpg", ".txt")):
            shutil.copy(os.path.join(root, f), f"{IMG_DIR}/{f}")
            n += 1
print("fichiers copies:", n, "->", IMG_DIR)
print(sorted(os.listdir(IMG_DIR))[:10])

In [ ]:
toml = f'''
[general]
shuffle_caption = true
caption_extension = ".txt"
keep_tokens = 1

[[datasets]]
resolution = {RESOLUTION}
batch_size = 2

  [[datasets.subsets]]
  image_dir = "{IMG_DIR}"
  class_tokens = "{TRIGGER_WORD}"
  num_repeats = {REPEATS}
'''
with open("/kaggle/working/dataset.toml", "w") as f:
    f.write(toml)
print(toml)

In [ ]:
!mkdir -p /kaggle/working/output
!accelerate launch --num_processes=1 --mixed_precision=fp16 --num_cpu_threads_per_process 2 train_network.py \
  --pretrained_model_name_or_path=/kaggle/working/models/v1-5-pruned-emaonly.safetensors \
  --dataset_config=/kaggle/working/dataset.toml \
  --output_dir=/kaggle/working/output \
  --output_name={PROJECT_NAME} \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim={NETWORK_DIM} \
  --network_alpha={NETWORK_ALPHA} \
  --learning_rate={LEARNING_RATE} \
  --max_train_epochs={EPOCHS} \
  --train_batch_size=2 \
  --mixed_precision=fp16 \
  --save_precision=fp16 \
  --optimizer_type=AdamW8bit \
  --lr_scheduler=cosine_with_restarts \
  --sdpa \
  --cache_latents \
  --gradient_checkpointing \
  --seed=42 \
  --logging_dir=/kaggle/working/logs

## Recuperer la LoRA

Le fichier final est `/kaggle/working/output/ambrelune_style.safetensors` (quelques dizaines de Mo).
Telecharge-le depuis le panneau *Output* de Kaggle (a droite), puis copie-le dans :

`C:\ComfyUI_windows_portable\ComfyUI\models\loras\ambrelune_style.safetensors`

Utilisation dans ComfyUI : ajoute un noeud `LoraLoader` entre le `CheckpointLoaderSimple` et le `KSampler`,
et mets `ambrelune style` en debut de chaque prompt positif.

In [ ]:
!ls -la /kaggle/working/output/